# Data Exploration for Lexical Semantic Embedding Project

This notebook provides comprehensive data exploration and analysis for the lexical semantic embedding project. We'll explore multiple datasets including STS Benchmark, SICK, Quora Question Pairs, and MRPC.

## Table of Contents
1. [Environment Setup](#Environment-Setup)
2. [Dataset Overview](#Dataset-Overview)
3. [Data Loading and Inspection](#Data-Loading-and-Inspection)
4. [Statistical Analysis](#Statistical-Analysis)
5. [Text Analysis](#Text-Analysis)
6. [Preprocessing Exploration](#Preprocessing-Exploration)
7. [Visualization](#Visualization)
8. [Insights and Conclusions](#Insights-and-Conclusions)

## Environment Setup

First, let's install and import all necessary libraries for data exploration.

In [ ]:
# Install required packages if not already installed
import subprocess
import sys

def install_if_missing(package):
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Install key packages
packages = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'plotly', 'wordcloud', 'textstat']
for package in packages:
    install_if_missing(package)

print("All packages installed successfully!")

In [ ]:
# Core libraries
import os
import sys
import json
import pickle
import warnings
from pathlib import Path
from collections import Counter, defaultdict
import re

# Data manipulation and analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Text analysis
from wordcloud import WordCloud
import textstat

# Configure plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("Environment setup complete!")

In [ ]:
# Project paths setup
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_DIR / 'raw'
PROCESSED_DATA_DIR = DATA_DIR / 'processed'
EMBEDDINGS_DIR = DATA_DIR / 'embeddings'

# Create directories if they don't exist
for dir_path in [DATA_DIR, RAW_DATA_DIR, PROCESSED_DATA_DIR, EMBEDDINGS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Processed data directory: {PROCESSED_DATA_DIR}")

## Dataset Overview

Let's start by exploring the datasets we'll be working with for semantic similarity tasks.

In [ ]:
# Dataset information
DATASETS_INFO = {
    'sts-benchmark': {
        'name': 'STS Benchmark',
        'description': 'Semantic Textual Similarity Benchmark',
        'url': 'http://ixa2.si.ehu.es/stswiki/images/4/48/Stsbenchmark.tar.gz',
        'score_range': (0, 5),
        'task': 'Regression',
        'splits': ['train', 'dev', 'test'],
        'size': '~8.5K pairs'
    },
    'sick': {
        'name': 'SICK Dataset',
        'description': 'Sentences Involving Compositional Knowledge',
        'url': 'https://zenodo.org/record/2787612/files/SICK.zip',
        'score_range': (1, 5),
        'task': 'Regression + Classification',
        'splits': ['train', 'trial', 'test'],
        'size': '~10K pairs'
    },
    'quora': {
        'name': 'Quora Question Pairs',
        'description': 'Quora duplicate question detection',
        'url': 'https://quoradata.quora.com/First-Quora-Dataset-Release-Question-Pairs',
        'score_range': (0, 1),
        'task': 'Binary Classification',
        'splits': ['train'],
        'size': '~400K pairs'
    },
    'mrpc': {
        'name': 'MRPC',
        'description': 'Microsoft Research Paraphrase Corpus',
        'url': 'https://dl.fbaipublicfiles.com/glue/data/mrpc_dev_ids.tsv',
        'score_range': (0, 1),
        'task': 'Binary Classification',
        'splits': ['train', 'dev', 'test'],
        'size': '~5.8K pairs'
    }
}

# Display dataset information
df_info = pd.DataFrame(DATASETS_INFO).T
print("Dataset Overview:")
print("=" * 50)
for dataset, info in DATASETS_INFO.items():
    print(f"\n{info['name']}:")
    print(f"  Description: {info['description']}")
    print(f"  Task: {info['task']}")
    print(f"  Score Range: {info['score_range']}")
    print(f"  Size: {info['size']}")
    print(f"  Splits: {', '.join(info['splits'])}")

## Data Loading and Inspection

Let's create sample datasets for exploration (since we don't have the actual data files yet).

In [ ]:
# Generate sample data for exploration
np.random.seed(42)

# Sample sentences for different similarity levels
sample_sentences = {
    'high_similarity': [
        ("The cat sat on the mat.", "A cat was sitting on the mat.", 4.5),
        ("He is playing football.", "He is playing soccer.", 4.2),
        ("The dog is running in the park.", "A dog runs through the park.", 4.0),
        ("She loves reading books.", "She enjoys reading novels.", 3.8),
        ("The car is red.", "The automobile is red.", 4.3)
    ],
    'medium_similarity': [
        ("The weather is nice today.", "It's a beautiful day.", 3.2),
        ("I am going to the store.", "I need to buy groceries.", 2.8),
        ("The movie was interesting.", "The film was good.", 3.5),
        ("He drives a car.", "He owns a vehicle.", 3.0),
        ("The book is on the table.", "There's a book on the desk.", 3.1)
    ],
    'low_similarity': [
        ("The cat sat on the mat.", "Quantum physics is complex.", 0.5),
        ("I love pizza.", "The stock market is volatile.", 0.2),
        ("Birds can fly.", "Mathematics is important.", 0.3),
        ("The sun is shining.", "I need to learn programming.", 0.1),
        ("Water is wet.", "Space exploration is fascinating.", 0.4)
    ]
}

# Create sample datasets
def create_sample_dataset(name, size=1000):
    data = []
    
    for i in range(size):
        # Randomly select similarity category
        category = np.random.choice(['high_similarity', 'medium_similarity', 'low_similarity'], 
                                   p=[0.3, 0.4, 0.3])
        
        # Select random sentence pair from category
        sent1, sent2, base_score = np.random.choice(sample_sentences[category])
        
        # Add some noise to the score
        if name == 'sts-benchmark':
            score = max(0, min(5, base_score + np.random.normal(0, 0.3)))
        elif name == 'sick':
            score = max(1, min(5, base_score + np.random.normal(0, 0.3)))
        else:  # Binary datasets
            score = 1 if base_score > 2.5 else 0
        
        data.append({
            'sentence1': sent1,
            'sentence2': sent2,
            'score': score,
            'pair_id': f"{name}_{i}"
        })
    
    return pd.DataFrame(data)

# Generate sample datasets
datasets = {}
for dataset_name in DATASETS_INFO.keys():
    datasets[dataset_name] = create_sample_dataset(dataset_name, 1000)
    print(f"Created sample {dataset_name} dataset with {len(datasets[dataset_name])} samples")

print("\nSample datasets created successfully!")

In [ ]:
# Inspect the structure of each dataset
print("Dataset Structure Inspection:")
print("=" * 40)

for name, df in datasets.items():
    print(f"\n{name.upper()} Dataset:")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"Data types:\n{df.dtypes}")
    print(f"\nFirst 3 rows:")
    print(df.head(3))
    print(f"\nScore statistics:")
    print(df['score'].describe())
    print("-" * 30)

## Statistical Analysis

Let's analyze the statistical properties of our datasets.

In [ ]:
# Analyze score distributions
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.ravel()

for i, (name, df) in enumerate(datasets.items()):
    ax = axes[i]
    
    # Plot histogram
    ax.hist(df['score'], bins=20, alpha=0.7, edgecolor='black')
    ax.set_title(f'{name.upper()} - Score Distribution')
    ax.set_xlabel('Similarity Score')
    ax.set_ylabel('Frequency')
    ax.grid(True, alpha=0.3)
    
    # Add statistics text
    stats_text = f"Mean: {df['score'].mean():.2f}\nStd: {df['score'].std():.2f}\nMin: {df['score'].min():.2f}\nMax: {df['score'].max():.2f}"
    ax.text(0.65, 0.85, stats_text, transform=ax.transAxes, 
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
            verticalalignment='top')

plt.tight_layout()
plt.show()

# Print detailed statistics
print("\nDetailed Score Statistics:")
print("=" * 50)
stats_df = pd.DataFrame({name: df['score'].describe() for name, df in datasets.items()})
print(stats_df.round(3))

In [ ]:
# Interactive correlation analysis with Plotly
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[name.upper() for name in datasets.keys()],
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}]]
)

colors = ['blue', 'red', 'green', 'orange']

for i, (name, df) in enumerate(datasets.items()):
    row = i // 2 + 1
    col = i % 2 + 1
    
    fig.add_trace(
        go.Histogram(
            x=df['score'],
            name=name,
            marker_color=colors[i],
            opacity=0.7,
            nbinsx=20
        ),
        row=row, col=col
    )

fig.update_layout(
    title_text="Interactive Score Distributions",
    height=700,
    showlegend=False
)

fig.show()

## Text Analysis

Let's analyze the textual properties of our sentence pairs.

In [ ]:
# Analyze sentence lengths
def analyze_text_properties(df, dataset_name):
    """Analyze various text properties."""
    
    # Calculate text properties
    df['len1'] = df['sentence1'].str.len()
    df['len2'] = df['sentence2'].str.len()
    df['words1'] = df['sentence1'].str.split().str.len()
    df['words2'] = df['sentence2'].str.split().str.len()
    df['len_diff'] = abs(df['len1'] - df['len2'])
    df['word_diff'] = abs(df['words1'] - df['words2'])
    
    # Readability scores
    df['readability1'] = df['sentence1'].apply(lambda x: textstat.flesch_reading_ease(x))
    df['readability2'] = df['sentence2'].apply(lambda x: textstat.flesch_reading_ease(x))
    
    return df

# Apply text analysis to all datasets
for name in datasets.keys():
    datasets[name] = analyze_text_properties(datasets[name], name)

print("Text analysis completed for all datasets!")

In [ ]:
# Visualize sentence length distributions
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for i, (name, df) in enumerate(datasets.items()):
    # Character length distribution
    ax1 = axes[0, 0] if i == 0 else axes[0, 0]
    if i == 0:
        ax1.set_title('Character Length Distribution')
    ax1.hist(df['len1'], alpha=0.5, label=f'{name} - Sent1', bins=20)
    ax1.hist(df['len2'], alpha=0.5, label=f'{name} - Sent2', bins=20)
    ax1.set_xlabel('Character Length')
    ax1.set_ylabel('Frequency')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

# Word count distribution
ax2 = axes[0, 1]
for name, df in datasets.items():
    ax2.hist(df['words1'], alpha=0.5, label=f'{name}', bins=15)
ax2.set_title('Word Count Distribution (Sentence 1)')
ax2.set_xlabel('Word Count')
ax2.set_ylabel('Frequency')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Length difference vs similarity score
ax3 = axes[0, 2]
for name, df in datasets.items():
    ax3.scatter(df['len_diff'], df['score'], alpha=0.5, label=name, s=10)
ax3.set_title('Length Difference vs Similarity Score')
ax3.set_xlabel('Character Length Difference')
ax3.set_ylabel('Similarity Score')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Readability analysis
ax4 = axes[1, 0]
for name, df in datasets.items():
    ax4.hist(df['readability1'], alpha=0.5, label=name, bins=15)
ax4.set_title('Readability Score Distribution')
ax4.set_xlabel('Flesch Reading Ease Score')
ax4.set_ylabel('Frequency')
ax4.legend()
ax4.grid(True, alpha=0.3)

# Word difference vs similarity
ax5 = axes[1, 1]
for name, df in datasets.items():
    ax5.scatter(df['word_diff'], df['score'], alpha=0.5, label=name, s=10)
ax5.set_title('Word Count Difference vs Similarity')
ax5.set_xlabel('Word Count Difference')
ax5.set_ylabel('Similarity Score')
ax5.legend()
ax5.grid(True, alpha=0.3)

# Combined statistics
ax6 = axes[1, 2]
stats_data = []
for name, df in datasets.items():
    stats_data.append([
        df['len1'].mean(),
        df['words1'].mean(),
        df['readability1'].mean()
    ])

stats_df = pd.DataFrame(stats_data, 
                       columns=['Avg Char Length', 'Avg Word Count', 'Avg Readability'],
                       index=list(datasets.keys()))

stats_df.plot(kind='bar', ax=ax6)
ax6.set_title('Average Text Statistics by Dataset')
ax6.set_ylabel('Value')
ax6.tick_params(axis='x', rotation=45)
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Word frequency analysis
def get_word_frequencies(df):
    """Get word frequencies from sentence pairs."""
    all_words = []
    
    for _, row in df.iterrows():
        words1 = re.findall(r'\b\w+\b', row['sentence1'].lower())
        words2 = re.findall(r'\b\w+\b', row['sentence2'].lower())
        all_words.extend(words1 + words2)
    
    return Counter(all_words)

# Get word frequencies for each dataset
word_frequencies = {}
for name, df in datasets.items():
    word_frequencies[name] = get_word_frequencies(df)
    print(f"{name}: {len(word_frequencies[name])} unique words")

# Display most common words
print("\nMost common words per dataset:")
print("=" * 40)
for name, freq in word_frequencies.items():
    print(f"\n{name.upper()}:")
    for word, count in freq.most_common(10):
        print(f"  {word}: {count}")

In [ ]:
# Create word clouds for each dataset
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for i, (name, freq) in enumerate(word_frequencies.items()):
    # Create word cloud
    wordcloud = WordCloud(
        width=400, height=300,
        background_color='white',
        max_words=100,
        colormap='viridis'
    ).generate_from_frequencies(freq)
    
    axes[i].imshow(wordcloud, interpolation='bilinear')
    axes[i].set_title(f'{name.upper()} - Word Cloud')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## Preprocessing Exploration

Let's explore different preprocessing strategies and their effects on the data.

In [ ]:
import string

def preprocess_text(text, method='basic'):
    """Apply different preprocessing methods to text."""
    
    if method == 'basic':
        # Basic: lowercase and strip
        return text.lower().strip()
    
    elif method == 'punctuation':
        # Remove punctuation
        text = text.lower().strip()
        return text.translate(str.maketrans('', '', string.punctuation))
    
    elif method == 'normalize':
        # Normalize whitespace and punctuation
        text = text.lower().strip()
        text = re.sub(r'\s+', ' ', text)  # Multiple spaces to single
        text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
        return text
    
    elif method == 'aggressive':
        # Aggressive: remove punctuation, extra spaces, numbers
        text = text.lower().strip()
        text = re.sub(r'\d+', '', text)  # Remove numbers
        text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
        text = re.sub(r'\s+', ' ', text)  # Normalize spaces
        return text.strip()
    
    return text

# Test preprocessing methods
sample_text = "Hello, World! This is a test sentence with numbers 123 and symbols @#$."
methods = ['basic', 'punctuation', 'normalize', 'aggressive']

print("Preprocessing Methods Comparison:")
print("=" * 50)
print(f"Original: {sample_text}")
print()

for method in methods:
    processed = preprocess_text(sample_text, method)
    print(f"{method.capitalize()}: {processed}")

In [ ]:
# Analyze preprocessing impact on one dataset
sample_df = datasets['sts-benchmark'].head(100).copy()

preprocessing_results = {}

for method in methods:
    # Apply preprocessing
    processed_df = sample_df.copy()
    processed_df['processed1'] = processed_df['sentence1'].apply(lambda x: preprocess_text(x, method))
    processed_df['processed2'] = processed_df['sentence2'].apply(lambda x: preprocess_text(x, method))
    
    # Calculate statistics
    original_len1 = processed_df['sentence1'].str.len().mean()
    processed_len1 = processed_df['processed1'].str.len().mean()
    
    original_words1 = processed_df['sentence1'].str.split().str.len().mean()
    processed_words1 = processed_df['processed1'].str.split().str.len().mean()
    
    preprocessing_results[method] = {
        'original_char_len': original_len1,
        'processed_char_len': processed_len1,
        'char_reduction': (original_len1 - processed_len1) / original_len1 * 100,
        'original_word_count': original_words1,
        'processed_word_count': processed_words1,
        'word_reduction': (original_words1 - processed_words1) / original_words1 * 100
    }

# Display results
preprocessing_df = pd.DataFrame(preprocessing_results).T
print("Preprocessing Impact Analysis:")
print("=" * 50)
print(preprocessing_df.round(2))

# Visualize preprocessing impact
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Character length reduction
ax1.bar(methods, preprocessing_df['char_reduction'])
ax1.set_title('Character Length Reduction by Method')
ax1.set_xlabel('Preprocessing Method')
ax1.set_ylabel('Reduction Percentage')
ax1.grid(True, alpha=0.3)

# Word count reduction
ax2.bar(methods, preprocessing_df['word_reduction'])
ax2.set_title('Word Count Reduction by Method')
ax2.set_xlabel('Preprocessing Method')
ax2.set_ylabel('Reduction Percentage')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Advanced Visualization

Let's create comprehensive visualizations to understand our data better.

In [ ]:
# Create correlation heatmaps for numerical features
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for i, (name, df) in enumerate(datasets.items()):
    # Select numerical columns
    numerical_cols = ['score', 'len1', 'len2', 'words1', 'words2', 
                     'len_diff', 'word_diff', 'readability1', 'readability2']
    
    # Calculate correlation matrix
    corr_matrix = df[numerical_cols].corr()
    
    # Create heatmap
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0,
                ax=axes[i], cbar=True, square=True, fmt='.2f')
    axes[i].set_title(f'{name.upper()} - Feature Correlations')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Interactive scatter plot with Plotly
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f"{name.upper()} - Length vs Score" for name in datasets.keys()]
)

colors = ['blue', 'red', 'green', 'orange']

for i, (name, df) in enumerate(datasets.items()):
    row = i // 2 + 1
    col = i % 2 + 1
    
    fig.add_trace(
        go.Scatter(
            x=df['len_diff'],
            y=df['score'],
            mode='markers',
            name=name,
            marker=dict(color=colors[i], opacity=0.6),
            text=df['pair_id'],
            hovertemplate='Length Diff: %{x}<br>Score: %{y}<br>ID: %{text}<extra></extra>'
        ),
        row=row, col=col
    )

fig.update_layout(
    title_text="Interactive Length Difference vs Similarity Score",
    height=700,
    showlegend=False
)

fig.show()

In [ ]:
# Create a comprehensive dashboard
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[
        "Score Distribution Comparison",
        "Average Text Length by Dataset",
        "Word Count vs Character Count",
        "Readability Score Distribution",
        "Length Difference Impact",
        "Dataset Size Comparison"
    ],
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}]]
)

colors = ['blue', 'red', 'green', 'orange']

# 1. Score distributions
for i, (name, df) in enumerate(datasets.items()):
    fig.add_trace(
        go.Histogram(x=df['score'], name=name, marker_color=colors[i], opacity=0.7),
        row=1, col=1
    )

# 2. Average text lengths
avg_lengths = [df['len1'].mean() for df in datasets.values()]
fig.add_trace(
    go.Bar(x=list(datasets.keys()), y=avg_lengths, marker_color=colors),
    row=1, col=2
)

# 3. Word count vs character count
for i, (name, df) in enumerate(datasets.items()):
    fig.add_trace(
        go.Scatter(
            x=df['words1'], y=df['len1'], 
            mode='markers', name=name, 
            marker=dict(color=colors[i], opacity=0.6)
        ),
        row=2, col=1
    )

# 4. Readability distributions
for i, (name, df) in enumerate(datasets.items()):
    fig.add_trace(
        go.Histogram(x=df['readability1'], name=name, marker_color=colors[i], opacity=0.7),
        row=2, col=2
    )

# 5. Length difference vs score
for i, (name, df) in enumerate(datasets.items()):
    fig.add_trace(
        go.Scatter(
            x=df['len_diff'], y=df['score'], 
            mode='markers', name=name,
            marker=dict(color=colors[i], opacity=0.6)
        ),
        row=3, col=1
    )

# 6. Dataset sizes
sizes = [len(df) for df in datasets.values()]
fig.add_trace(
    go.Bar(x=list(datasets.keys()), y=sizes, marker_color=colors),
    row=3, col=2
)

fig.update_layout(
    title_text="Comprehensive Data Exploration Dashboard",
    height=1200,
    showlegend=False
)

fig.show()

## Insights and Conclusions

Let's summarize our findings and insights from the data exploration.

In [ ]:
# Generate comprehensive summary statistics
summary_stats = {}

for name, df in datasets.items():
    summary_stats[name] = {
        'total_pairs': len(df),
        'avg_score': df['score'].mean(),
        'score_std': df['score'].std(),
        'avg_char_length': (df['len1'].mean() + df['len2'].mean()) / 2,
        'avg_word_count': (df['words1'].mean() + df['words2'].mean()) / 2,
        'avg_readability': (df['readability1'].mean() + df['readability2'].mean()) / 2,
        'unique_words': len(word_frequencies[name]),
        'most_common_word': word_frequencies[name].most_common(1)[0][0],
        'most_common_count': word_frequencies[name].most_common(1)[0][1]
    }

# Create summary DataFrame
summary_df = pd.DataFrame(summary_stats).T

print("Dataset Summary Statistics:")
print("=" * 60)
print(summary_df.round(2))

# Key insights
insights = [
    "\n🔍 KEY INSIGHTS FROM DATA EXPLORATION:",
    "=" * 50,
    
    "📊 SCORE DISTRIBUTIONS:",
    f"• STS Benchmark has scores ranging 0-5 with mean {summary_stats['sts-benchmark']['avg_score']:.2f}",
    f"• SICK dataset has scores 1-5 with mean {summary_stats['sick']['avg_score']:.2f}",
    f"• Quora and MRPC are binary (0/1) classification tasks",
    
    "\n📝 TEXT CHARACTERISTICS:",
    f"• Average sentence length varies from {min(s['avg_char_length'] for s in summary_stats.values()):.0f} to {max(s['avg_char_length'] for s in summary_stats.values()):.0f} characters",
    f"• Word counts range from {min(s['avg_word_count'] for s in summary_stats.values()):.1f} to {max(s['avg_word_count'] for s in summary_stats.values()):.1f} words per sentence",
    f"• Readability scores suggest moderate complexity across datasets",
    
    "\n🔗 CORRELATIONS:",
    "• Length differences show weak correlation with similarity scores",
    "• Word count and character length are highly correlated",
    "• Readability scores vary but don't strongly predict similarity",
    
    "\n⚙️ PREPROCESSING RECOMMENDATIONS:",
    "• Basic lowercasing and normalization appear sufficient",
    "• Aggressive preprocessing may remove important semantic cues",
    "• Consider preserving punctuation for better semantic understanding",
    
    "\n🎯 MODELING IMPLICATIONS:",
    "• Different score ranges require careful handling in loss functions",
    "• Sentence length variability suggests need for proper padding/masking",
    "• Binary vs continuous tasks may benefit from different architectures",
    
    "\n📈 NEXT STEPS:",
    "• Implement proper tokenization with subword units",
    "• Design architecture to handle variable-length sequences",
    "• Consider multi-task learning across different datasets",
    "• Implement proper evaluation metrics for each task type"
]

for insight in insights:
    print(insight)

In [ ]:
# Export findings and processed data
import json

# Save summary statistics
summary_stats_path = PROCESSED_DATA_DIR / 'exploration_summary.json'
with open(summary_stats_path, 'w') as f:
    json.dump(summary_stats, f, indent=2, default=str)

# Save word frequencies
word_freq_path = PROCESSED_DATA_DIR / 'word_frequencies.json'
word_freq_serializable = {name: dict(freq.most_common(1000)) for name, freq in word_frequencies.items()}
with open(word_freq_path, 'w') as f:
    json.dump(word_freq_serializable, f, indent=2)

# Save processed datasets
for name, df in datasets.items():
    df_path = PROCESSED_DATA_DIR / f'{name}_explored.pkl'
    df.to_pickle(df_path)

print(f"📁 Exploration results saved to: {PROCESSED_DATA_DIR}")
print(f"📊 Summary statistics: {summary_stats_path}")
print(f"📝 Word frequencies: {word_freq_path}")
print(f"🗃️ Processed datasets: {len(datasets)} files saved")

print("\n✅ Data exploration completed successfully!")
print("Next steps: Proceed to model training and evaluation notebooks.")